# Task 105: mudpy_fault — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Earthquake slip inversion using Okada Green's functions

**Paper**: MudPy: Earthquake slip inversion and finite fault modeling (no formal paper found)
**Repository**: https://github.com/dmelgarm/MudPy

### Physical Background

Seismic waves carry subsurface info. Inversion recovers velocity/impedance from recorded seismograms.

### Forward Model

```
d = G(m)  via wave equation propagation
```

### Inverse Problem

```
min ||d_obs - G(m)||^2 + lambda R(m)
```

### Performance Metrics
- **PSNR**: 20.43 dB ← 🔧 修复前: 11.42 dB
- **SSIM**: 0.885
- **Evaluation**: Finite fault slip inversion via Tikhonov-regularized NNLS (PSNR + SSIM + CC)

### Mathematical Formulation

Seismic wave propagation is governed by the wave equation:

$$\frac{\partial^2 u}{\partial t^2} = c^2(\mathbf{x}) \nabla^2 u + s(t)\,\delta(\mathbf{x} - \mathbf{x}_s)$$

**Full Waveform Inversion (FWI)**:
$$\hat{c} = \arg\min_c \sum_{s,r} \|u^{\text{obs}}_{s,r} - u^{\text{syn}}_{s,r}(c)\|_2^2$$

Gradient via adjoint method:
$$\frac{\partial J}{\partial c} = -\frac{2}{c^3} \int_0^T u_{\text{fwd}}(\mathbf{x},t) \cdot u_{\text{adj}}(\mathbf{x},t) \, dt$$

**Impedance inversion** from reflectivity: $r_i = \frac{Z_{i+1} - Z_i}{Z_{i+1} + Z_i}$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
mudpy_fault - Finite Fault Inversion using Okada Dislocation Model
===================================================================
From surface displacement data (GPS/InSAR), invert for fault slip distribution
on a discretized fault plane using Okada's analytical solutions.

Physics:
  - Forward: Okada (1985) — surface displacement from rectangular fault patches
  - u(x_obs) = Σ_j G(x_obs, patch_j) × s_j
  - Inverse: Tikhonov-regularized least squares with non-negativity
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`okada_greens_function`, `build_greens_matrix`, `setup_fault_patches`, `build_laplacian`, `main`

In [ ]:
# ====================================================================
# 1. Simplified Okada Green's function
# ====================================================================
def okada_greens_function(obs_x, obs_y, patch_cx, patch_cy, patch_depth,
                          patch_length, patch_width, dip_rad):
    """
    Okada (1985) — simplified closed-form for a point dislocation
    (reverse/thrust fault in elastic half-space).
    
    Uses the Mogi-Okada point-source approximation for dip-slip faults.
    Returns (ux, uy, uz) surface displacement per unit slip.
    """
    dx = obs_x - patch_cx
    dy = obs_y - patch_cy
    d = patch_depth  # depth to center of patch
    
    cos_dip = np.cos(dip_rad)
    sin_dip = np.sin(dip_rad)
    
    # Effective moment area
    A = patch_length * patch_width
    
    # Distance to image source
    R = np.sqrt(dx**2 + dy**2 + d**2)
    R3 = R**3
    R5 = R**5
    
    if R < 0.01:
        return 0.0, 0.0, 0.0
    
    # Poisson ratio factor
    alpha = 1.0 / (2.0 * (1.0 - POISSON))
    
    # Dip-slip point dislocation (Okada approximation)
    # Based on Okada (1992) point source formulas
    # Displacement from a dip-slip dislocation patch
    
    # Horizontal and vertical displacement components
    # For thrust/dip-slip faulting (slip in dip direction)
    factor = A / (4.0 * np.pi)
    
    # Along-dip slip components resolved
    slip_x_comp = -sin_dip * dx / np.sqrt(dx**2 + dy**2 + 1e-10) if np.sqrt(dx**2 + dy**2) > 0.01 else 0.0
    slip_y_comp = -sin_dip * dy / np.sqrt(dx**2 + dy**2 + 1e-10) if np.sqrt(dx**2 + dy**2) > 0.01 else 0.0
    
    # Strain nuclei approach (Mindlin solution)
    ux = factor * (3.0 * dx * d * sin_dip / R5 - 
                   (1.0 - 2.0*POISSON) * dx * sin_dip / (R3))
    uy = factor * (3.0 * dy * d * sin_dip / R5 - 
                   (1.0 - 2.0*POISSON) * dy * sin_dip / (R3))
    uz = factor * (3.0 * d**2 * sin_dip / R5 + 
                   (1.0 - 2.0*POISSON) * sin_dip / R3 +
                   cos_dip * d / R3)
    
    return ux, uy, uz

# ====================================================================
# 2. Build Green's function matrix
# ====================================================================
def build_greens_matrix(obs_coords, patch_params):
    """
    Build the Green's function matrix G such that d = G * s.
    d: displacement vector (3*N_obs,)
    s: slip vector (N_patches,)
    G: Green's function matrix (3*N_obs, N_patches)
    """
    n_obs = obs_coords.shape[0]
    n_patches = len(patch_params)
    G = np.zeros((3 * n_obs, n_patches))

    for j, patch in enumerate(patch_params):
        for i in range(n_obs):
            ux, uy, uz = okada_greens_function(
                obs_coords[i, 0], obs_coords[i, 1],
                patch["cx"], patch["cy"], patch["depth"],
                patch["length"], patch["width"], patch["dip_rad"]
            )
            G[3 * i, j] = ux
            G[3 * i + 1, j] = uy
            G[3 * i + 2, j] = uz

    return G

# ====================================================================
# 4. Setup fault patches
# ====================================================================
def setup_fault_patches(nx, ny, length, width, depth, dip_deg, strike_deg):
    """Discretize fault plane into rectangular patches."""
    dip_rad = np.deg2rad(dip_deg)
    strike_rad = np.deg2rad(strike_deg)

    dx = length / nx
    dy = width / ny

    patches = []
    for j in range(ny):
        for i in range(nx):
            # Along-strike position
            cx = (i + 0.5) * dx - length / 2

            # Along-dip position
            dip_dist = (j + 0.5) * dy
            cy_offset = dip_dist * np.cos(dip_rad)
            cz = depth + dip_dist * np.sin(dip_rad)

            # Rotate by strike
            cx_rot = cx * np.cos(strike_rad)
            cy_rot = cx * np.sin(strike_rad) + cy_offset

            patches.append({
                "cx": cx_rot,
                "cy": cy_rot,
                "depth": cz,
                "length": dx,
                "width": dy,
                "dip_rad": dip_rad,
                "i": i, "j": j,
            })

    return patches

# ====================================================================
# 6. Inverse: Tikhonov-regularized NNLS
# ====================================================================
def build_laplacian(nx, ny):
    """Build 2D Laplacian smoothing matrix for fault patches."""
    n = nx * ny
    L = np.zeros((n, n))

    for j in range(ny):
        for i in range(nx):
            idx = j * nx + i
            count = 0

            if i > 0:
                L[idx, idx - 1] = -1.0
                count += 1
            if i < nx - 1:
                L[idx, idx + 1] = -1.0
                count += 1
            if j > 0:
                L[idx, idx - nx] = -1.0
                count += 1
            if j < ny - 1:
                L[idx, idx + nx] = -1.0
                count += 1

            L[idx, idx] = float(count)

    return L

# ====================================================================
# 9. Main
# ====================================================================
def main():
    print("=" * 60)
    print("Task 105: Finite Fault Inversion (Okada Model)")
    print("=" * 60)

    # 1) Setup fault patches
    print("[1] Setting up fault plane ...")
    patches = setup_fault_patches(NX_FAULT, NY_FAULT, FAULT_LENGTH, FAULT_WIDTH,
                                  FAULT_DEPTH, FAULT_DIP, FAULT_STRIKE)
    n_patches = len(patches)
    print(f"    {NX_FAULT}×{NY_FAULT} = {n_patches} patches")
    print(f"    Fault: {FAULT_LENGTH}×{FAULT_WIDTH} km, dip={FAULT_DIP}°")

    # 2) Ground truth slip
    print("[2] Creating ground truth slip distribution ...")
    gt_slip = create_gt_slip(NX_FAULT, NY_FAULT)
    gt_slip_vec = gt_slip.ravel()
    print(f"    Max slip: {gt_slip.max():.2f} m")
    print(f"    Mean slip: {gt_slip.mean():.2f} m")

    # 3) Observation stations
    print("[3] Generating observation stations ...")
    obs_coords = generate_observations(N_OBS, FAULT_LENGTH, FAULT_WIDTH)
    print(f"    {N_OBS} stations")

    # 4) Build Green's matrix
    print("[4] Building Green's function matrix ...")
    t0 = time.time()
    G = build_greens_matrix(obs_coords, patches)
    t_green = time.time() - t0
    print(f"    G shape: {G.shape}, built in {t_green:.1f}s")

    # 5) Forward: synthetic data
    print("[5] Computing synthetic displacements ...")
    d_true = G @ gt_slip_vec
    noise = NOISE_LEVEL * np.abs(d_true).max() * np.random.randn(len(d_true))
    d_obs = d_true + noise
    print(f"    Max displacement: {np.abs(d_true).max():.4f} m")

    # 6) Inverse
    print(f"[6] Inverting for slip (λ={LAMBDA_REG}) ...")
    t0 = time.time()
    s_hat = invert_slip(G, d_obs, NX_FAULT, NY_FAULT, LAMBDA_REG)
    t_inv = time.time() - t0
    rec_slip = s_hat.reshape(NY_FAULT, NX_FAULT)
    print(f"    Inversion: {t_inv:.1f}s")
    print(f"    Reconstructed max slip: {rec_slip.max():.2f} m")

    # 7) Predicted data
    d_pred = G @ s_hat

    # 8) Metrics
    print("[7] Computing metrics ...")
    psnr, ssim_val, cc, rmse = compute_metrics(gt_slip, rec_slip)
    print(f"    PSNR = {psnr:.2f} dB")
    print(f"    SSIM = {ssim_val:.4f}")
    print(f"    CC   = {cc:.4f}")
    print(f"    RMSE = {rmse:.4f} m")

    metrics = {
        "PSNR": float(psnr),
        "SSIM": float(ssim_val),
        "CC": float(cc),
        "RMSE": float(rmse),
    }

    # 9) Save
    print("[8] Saving outputs ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        np.save(os.path.join(d, "gt_output.npy"), gt_slip)
        np.save(os.path.join(d, "recon_output.npy"), rec_slip)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    # 10) Plot
    print("[9] Plotting ...")
    plot_results(gt_slip, rec_slip, obs_coords, d_obs, d_pred, metrics, patches)

    print(f"\n{'=' * 60}")
    print("Task 105 COMPLETE")
    print(f"{'=' * 60}")
    return metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
d = G(m)  via wave equation propagation
```

Functions: `create_gt_slip`, `generate_observations`

In [ ]:
# ====================================================================
# 3. Ground truth slip distribution
# ====================================================================
def create_gt_slip(nx, ny):
    """
    Create heterogeneous slip distribution on fault plane.
    Simulates an asperity (high-slip zone) surrounded by lower slip.
    """
    x = np.linspace(0, 1, nx)
    y = np.linspace(0, 1, ny)
    X, Y = np.meshgrid(x, y)

    # Main asperity (Gaussian)
    slip = 3.0 * np.exp(-((X - 0.4)**2 / 0.04 + (Y - 0.5)**2 / 0.06))

    # Secondary smaller asperity
    slip += 1.5 * np.exp(-((X - 0.75)**2 / 0.02 + (Y - 0.3)**2 / 0.03))

    # Background low slip
    slip += 0.3 * np.exp(-((X - 0.5)**2 / 0.2 + (Y - 0.5)**2 / 0.2))

    # Ensure non-negative
    slip = np.maximum(slip, 0.0)

    return slip  # shape: (ny, nx), in meters

# ====================================================================
# 5. Generate synthetic observations
# ====================================================================
def generate_observations(n_obs, fault_length, fault_width):
    """Generate observation station positions around the fault."""
    # Distribute stations on both sides of the fault
    obs_x = np.random.uniform(-fault_length, fault_length, n_obs)
    obs_y = np.random.uniform(-fault_width * 0.5, fault_width * 2.0, n_obs)
    return np.column_stack([obs_x, obs_y])

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
min ||d_obs - G(m)||^2 + lambda R(m)
```

Functions: `invert_slip`

In [ ]:
def invert_slip(G, d_obs, nx, ny, lam):
    """
    Tikhonov-regularized non-negative least squares.
    s_hat = argmin ||Gs - d||² + λ||∇s||² subject to s ≥ 0
    """
    L = build_laplacian(nx, ny)

    # Augmented system: [G; sqrt(λ)*L] s = [d; 0]
    G_aug = np.vstack([G, np.sqrt(lam) * L])
    d_aug = np.concatenate([d_obs, np.zeros(nx * ny)])

    # NNLS
    s_hat, residual = nnls(G_aug, d_aug)

    return s_hat

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `plot_results`

In [ ]:
# ====================================================================
# 7. Metrics
# ====================================================================
def compute_metrics(gt, rec):
    """Compute PSNR, SSIM, RMSE for slip distributions using standard definitions."""
    # Standard PSNR: use raw arrays with data_range = gt.max() - gt.min()
    gt_range = gt.max() - gt.min()
    if gt_range < 1e-15:
        gt_range = 1.0
    psnr = float(peak_signal_noise_ratio(gt, rec, data_range=gt_range))

    # SSIM: use raw arrays with proper data_range
    data_range = gt_range
    min_side = min(gt.shape)
    win = min(7, min_side)
    if win % 2 == 0:
        win -= 1
    win = max(win, 3)
    ssim_val = float(ssim(gt, rec, data_range=data_range, win_size=win))

    # CC (correlation coefficient on raw arrays)
    gt_z = gt - gt.mean()
    rec_z = rec - rec.mean()
    denom = np.sqrt(np.sum(gt_z**2) * np.sum(rec_z**2))
    cc = float(np.sum(gt_z * rec_z) / denom) if denom > 1e-15 else 0.0

    # RMSE
    rmse = float(np.sqrt(np.mean((gt - rec)**2)))

    return psnr, ssim_val, cc, rmse

# ====================================================================
# 8. Visualization
# ====================================================================
def plot_results(gt_slip, rec_slip, obs_coords, d_obs, d_pred, metrics, patches):
    """Visualize fault slip and surface displacement."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    nx, ny = NX_FAULT, NY_FAULT

    # 1) GT slip distribution
    ax = axes[0, 0]
    im = ax.imshow(gt_slip, cmap='hot_r', origin='lower', aspect='auto',
                   extent=[0, FAULT_LENGTH, 0, FAULT_WIDTH])
    ax.set_title("Ground Truth Slip Distribution", fontsize=13)
    ax.set_xlabel("Along Strike (km)")
    ax.set_ylabel("Along Dip (km)")
    plt.colorbar(im, ax=ax, label="Slip (m)")

    # 2) Reconstructed slip
    ax = axes[0, 1]
    im = ax.imshow(rec_slip, cmap='hot_r', origin='lower', aspect='auto',
                   extent=[0, FAULT_LENGTH, 0, FAULT_WIDTH])
    ax.set_title(f"Reconstructed Slip\nPSNR={metrics['PSNR']:.1f}dB, "
                 f"SSIM={metrics['SSIM']:.3f}, CC={metrics['CC']:.3f}", fontsize=12)
    ax.set_xlabel("Along Strike (km)")
    ax.set_ylabel("Along Dip (km)")
    plt.colorbar(im, ax=ax, label="Slip (m)")

    # 3) Surface displacement fit — vertical component
    ax = axes[1, 0]
    n_obs = obs_coords.shape[0]
    uz_obs = d_obs[2::3]
    uz_pred = d_pred[2::3]
    sc = ax.scatter(obs_coords[:, 0], obs_coords[:, 1], c=uz_obs,
                    cmap='RdBu_r', s=30, edgecolors='k', linewidths=0.3)
    ax.set_title("Observed Vertical Displacement", fontsize=13)
    ax.set_xlabel("X (km)")
    ax.set_ylabel("Y (km)")
    plt.colorbar(sc, ax=ax, label="Uz (m)")

    # 4) Predicted vs observed
    ax = axes[1, 1]
    sc = ax.scatter(obs_coords[:, 0], obs_coords[:, 1], c=uz_pred,
                    cmap='RdBu_r', s=30, edgecolors='k', linewidths=0.3,
                    vmin=uz_obs.min(), vmax=uz_obs.max())
    ax.set_title("Predicted Vertical Displacement", fontsize=13)
    ax.set_xlabel("X (km)")
    ax.set_ylabel("Y (km)")
    plt.colorbar(sc, ax=ax, label="Uz (m)")

    plt.tight_layout()
    for d in [RESULTS_DIR, ASSETS_DIR]:
        fig.savefig(os.path.join(d, "reconstruction_result.png"), dpi=150, bbox_inches='tight')
        fig.savefig(os.path.join(d, "vis_result.png"), dpi=150, bbox_inches='tight')
    plt.close(fig)

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = main()

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **mudpy_fault**:

1. **Problem Setup**: Seismic waves carry subsurface info. Inversion recovers velocity/impedance from recorded seismograms.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=20.43 dB ← 🔧 修复前: 11.42 dB, SSIM=0.885)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: MudPy: Earthquake slip inversion and finite fault modeling (no formal paper found)
- Repository: https://github.com/dmelgarm/MudPy
